In [ ]:
from transformers import RobertaTokenizerFast, RobertaForSequenceClassification, TrainingArguments, Trainer
from datasets import Dataset
import torch
import evaluate
import numpy as np
import pandas as pd

In [ ]:
df = pd.read_csv("2022_Patient_level_linkage_withIDs.csv", dtype="string")

In [ ]:
df['Match Status'] = df['Match Status'].replace({'Match': '1', 'Non-Match': '0'})
df['Match Status'] = df['Match Status'].astype(int)
df.rename(columns={'Match Status': 'labels'}, inplace=True)

In [ ]:
base_model_name = "roberta-base"


tokenizer = RobertaTokenizerFast.from_pretrained(base_model_name)


special_tokens_dict = {
    "additional_special_tokens": ["[COL]", "[VAL]"]
}
tokenizer.add_special_tokens(special_tokens_dict)

print("Added special tokens:", tokenizer.additional_special_tokens)


In [ ]:
def serialize_record(record_dict):

    serialized_str = ""
    for col_name, value in record_dict.items():
        serialized_str += f" [COL] {col_name} [VAL] {value}"
    return serialized_str.strip()

In [ ]:
def create_input_pair(row):
    record1_dict = {
        "First Name": row["record1 First Name"],
        "Middle Name": row["record1 Middle Name"],
        "Last Name": row["record1 Last Name"],
        "Date of Birth": row["record1 Date of Birth"],
        "SSN": row["record1 SSN"],
        "Sex": row["record1 Sex"],
        "Address": row["record1 Address"],
    }
    record2_dict = {
        "First Name": row["record2 First Name"],
        "Middle Name": row["record2 Middle Name"],
        "Last Name": row["record2 Last Name"],
        "Date of Birth": row["record2 Date of Birth"],
        "SSN": row["record2 SSN"],
        "Sex": row["record2 Sex"],
        "Address": row["record2 Address"],
    }
    
    serialized_r1 = serialize_record(record1_dict)
    serialized_r2 = serialize_record(record2_dict)
    
    return serialized_r1, serialized_r2


In [ ]:
def tokenize_function(examples):

    record1_texts = []
    record2_texts = []
    
    for i in range(len(examples["labels"])):
        row = {
            "record1 First Name": examples["record1 First Name"][i],
            "record1 Middle Name": examples["record1 Middle Name"][i],
            "record1 Last Name": examples["record1 Last Name"][i],
            "record1 Date of Birth": examples["record1 Date of Birth"][i],
            "record1 SSN": examples["record1 SSN"][i],
            "record1 Sex": examples["record1 Sex"][i],
            "record1 Address": examples["record1 Address"][i],
            "record2 First Name": examples["record2 First Name"][i],
            "record2 Middle Name": examples["record2 Middle Name"][i],
            "record2 Last Name": examples["record2 Last Name"][i],
            "record2 Date of Birth": examples["record2 Date of Birth"][i],
            "record2 SSN": examples["record2 SSN"][i],
            "record2 Sex": examples["record2 Sex"][i],
            "record2 Address": examples["record2 Address"][i],
        }
        
        r1, r2 = create_input_pair(row)
        record1_texts.append(r1)
        record2_texts.append(r2)
    

    tokenized_batch = tokenizer(
        record1_texts,
        record2_texts,
        padding="max_length",
        truncation=True,
        max_length=256,
    )
    return tokenized_batch


In [ ]:
from sklearn.model_selection import train_test_split
from datasets import Dataset


train_dataset = Dataset.from_pandas(df)


train_dataset = train_dataset.map(tokenize_function, batched=True)


columns_to_remove = [
    "record1 First Name", "record1 Middle Name", "record1 Last Name",
    "record1 Date of Birth", "record1 SSN", "record1 Sex", "record1 Address",
    "record2 First Name", "record2 Middle Name", "record2 Last Name",
    "record2 Date of Birth", "record2 SSN", "record2 Sex", "record2 Address"
]

train_dataset = train_dataset.remove_columns(columns_to_remove)


train_dataset.set_format("torch")


In [ ]:
model = RobertaForSequenceClassification.from_pretrained(
    base_model_name,
    num_labels=2
)


model.resize_token_embeddings(len(tokenizer))


In [ ]:
from transformers import TrainingArguments

batch_size = 32

training_args = TrainingArguments(
    output_dir="roberta-linkage-classifier-checkpoints",
    save_strategy="epoch",
    learning_rate=2e-5,  
    per_device_train_batch_size=batch_size,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=100,
    lr_scheduler_type="cosine",
)


In [ ]:
accuracy_metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return accuracy_metric.compute(predictions=preds, references=labels)


In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    processing_class=tokenizer,  
    compute_metrics=compute_metrics,
)


In [ ]:
trainer.train()

In [ ]:
model.save_pretrained('roberta-linkage-classifier')
tokenizer.save_pretrained('roberta-linkage-classifier')